# RHF -> UHF -> BS-UHF -> LUCJ -> Local SQD Recovery Walkthrough

This notebook runs the same **RHF -> UHF -> broken-symmetry-UHF (BS-UHF) -> LUCJ circuit -> SQD
recovery** pipeline this repo's `sbd` package runs at scale on Fugaku/ROQUO -- end to end, on a
laptop, in well under a few minutes. No HPC, no Prefect server, no real IBM Quantum hardware.

The pipeline is molecule-agnostic: it consumes any FCIDUMP (the standard PySCF/Molpro integral-dump
format: an `&FCI NORB=.. NELEC=.. MS2=..` header plus integrals) and runs the same RHF/UHF/BS-UHF
code path regardless of which molecule produced it. This notebook happens to use a real Fe2S2
iron-sulfur-cluster FCIDUMP (see the note before Step 0), but nothing below is iron-specific --
point Step 0 at your own FCIDUMP and the rest of the notebook is unchanged.

This is the no-HPC companion to:

- [`../../docs/tutorials/run_uhf_bsuhf_any_molecule.md`](../../docs/tutorials/run_uhf_bsuhf_any_molecule.md)
  -- the full tutorial, including how to scale this up to a genuine multi-node, multi-step recovery
  run on Fugaku or ROQUO.
- [`run_recover.py`](run_recover.py) and [`README.md`](README.md) in this same directory -- the
  production-scale CLI this notebook's mechanics feed into.


In [1]:
import contextlib
import io
import json
import logging
import os
import re
import time
import urllib.request
from pathlib import Path

import numpy as np

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S",
)
log = logging.getLogger("walkthrough")

import prefect.logging.loggers as _plg
_orig_get_run_logger = _plg.get_run_logger
def _safe_get_run_logger(*a, **kw):
    try:
        return _orig_get_run_logger(*a, **kw)
    except Exception:
        return log
_plg.get_run_logger = _safe_get_run_logger
import prefect.logging as _pl
_pl.get_run_logger = _safe_get_run_logger
from qcsc_workflow_utility import chem as _chem
_chem.get_run_logger = _safe_get_run_logger
from sbd import solver_job as _sj
_sj.get_run_logger = _safe_get_run_logger

import ffsim
from qcsc_workflow_utility.chem import compute_molecular_integrals_from_fcidump
from qiskit_addon_sqd.configuration_recovery import recover_configurations
from qiskit_addon_sqd.counts import bit_array_to_arrays, generate_bit_array_uniform
from sbd.lucj import create_lucj_circuit, initialize_ucj_parameters
from sbd.sqd import _compute_excitation_counts, postselect_bitstrings, subsample_open_shell

t_notebook_start = time.perf_counter()
print("Setup complete: Prefect get_run_logger patched to fall back to stdlib logging.")


13:09:41 [INFO] Applied prefect_qiskit twirling patch (sampler Options now carries `twirling`).


Setup complete: Prefect get_run_logger patched to fall back to stdlib logging.


## Step 0 -- Get an FCIDUMP

This example uses a real Fe2S2 iron-sulfur cluster FCIDUMP because it's a genuine, publicly
available, strongly-correlated open-shell system this repo already validates against -- swap in
your own FCIDUMP here; nothing below is iron-specific.


In [2]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FCIDUMP_FILE = DATA_DIR / "fe2s2.fcidump"
FCIDUMP_URL = (
    "https://raw.githubusercontent.com/zhendongli2008/"
    "Active-space-model-for-Iron-Sulfur-Clusters/main/Fe2S2_and_Fe4S4/Fe2S2/fe2s2"
)

if not FCIDUMP_FILE.exists():
    urllib.request.urlretrieve(FCIDUMP_URL, FCIDUMP_FILE)

header_line = FCIDUMP_FILE.read_text().splitlines()[0]
norb = int(re.search(r"NORB=\s*(\d+)", header_line).group(1))
nelec = int(re.search(r"NELEC=\s*(\d+)", header_line).group(1))
ms2 = int(re.search(r"MS2=\s*(-?\d+)", header_line).group(1))
assert (norb, nelec, ms2) == (20, 30, 0), (norb, nelec, ms2)
print(f"FCIDUMP: {FCIDUMP_FILE}")
print(f"Header : NORB={norb}  NELEC={nelec}  MS2={ms2}")


FCIDUMP: /Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/examples/sbd_uhf_recover_any_molecule/data/fe2s2.fcidump
Header : NORB=20  NELEC=30  MS2=0


## Step 1 -- RHF baseline

`compute_molecular_integrals_from_fcidump` is a Prefect `@task`. Outside a flow run we call its
raw function via `.fn(...)` -- the same pattern `run_local.py` / `run_local_uhf.py` use to drive
this code with no Prefect orchestration at all. With `unrestricted=False` it runs a closed-shell
RHF, then a CCSD on top (the amplitude seed the LUCJ ansatz needs later), directly on the FCIDUMP
integrals.

The returned `ElectronicProperties` object carries integrals/occupancies/`t2` but not the
underlying PySCF mean-field object's `e_tot` or `spin_square()`, so we capture those with a small
monkeypatch on `chem._build_property(_uhf)` -- it runs no extra SCF/CCSD, it just reads `mf.e_tot`
off the same mean-field object the task already built before its result is thrown away.


In [3]:
_orig_build_property = _chem._build_property
_captured = {}

def _capture_build_property(mf, norb_, spin_sq, buf):
    _captured["rhf_e_tot"] = float(mf.e_tot)
    return _orig_build_property(mf, norb_, spin_sq, buf)

_chem._build_property = _capture_build_property

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    ep_rhf = compute_molecular_integrals_from_fcidump.fn(str(FCIDUMP_FILE), unrestricted=False)
e_rhf = _captured["rhf_e_tot"]
print(f"RHF energy: {e_rhf:.6f} Ha")


Overwritten attributes  get_hcore get_ovlp  of <class 'pyscf.soscf.newton_ah.SecondOrderSymAdaptedRHF'>
<class 'pyscf.soscf.newton_ah.SecondOrderSymAdaptedRHF'> does not have attributes  symmetry
Overwritten attributes  get_hcore get_ovlp  of <class 'pyscf.scf.hf_symm.SymAdaptedRHF'>
converged SCF energy = -116.205816114879


13:09:42 [INFO] Freeze 0 electrons in irreps []
    30 free electrons in irreps IR1

******** <class 'pyscf.cc.ccsd.CCSD'> ********
CC2 = 0
CCSD nocc = 15, nmo = 20
max_cycle = 200
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-05
diis_space = 12
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 0 MB)
Init t2, MP2 energy = -116.322042751183  E_corr(MP2) -0.116226
... [28267 characters of PySCF internal SCF/stability-analysis/Newton-solver log output suppressed for readability -- this is expected verbose=4 chatter, not an error] ...
107896
cycle = 197  norm(lambda1,lambda2) = 0.00107924
cycle = 198  norm(lambda1,lambda2) = 0.00107898
cycle = 199  norm(lambda1,lambda2) = 0.00107889
cycle = 200  norm(lambda1,lambda2) = 0.00107903



RHF energy: -116.205816 Ha


## Step 2 -- UHF, energy-ordered (no localization)

Same function, `unrestricted=True`, with no `FE4S4_AF_GROUPS` set. For a formally closed-shell
system (`MS2=0`) like this one, plain UHF started from the RHF density is a stationary point of
the UHF equations and stays *at* the RHF solution unless the initial guess is seeded with a
spin-localized (broken-symmetry) guess -- Step 3 below. So this energy is often numerically very
close to (or identical to) the RHF energy above; that is the expected, correct behavior, not a
bug.


In [4]:
_orig_build_property_uhf = _chem._build_property_uhf

def _capture_build_property_uhf(key):
    def _fn(mf, norb_, spin_sq, buf):
        _captured[f"{key}_e_tot"] = float(mf.e_tot)
        try:
            _captured[f"{key}_spin_sq"] = float(mf.spin_square()[0])
        except Exception:
            _captured[f"{key}_spin_sq"] = None
        return _orig_build_property_uhf(mf, norb_, spin_sq, buf)
    return _fn

os.environ.pop("FE4S4_AF_GROUPS", None)
_chem._build_property_uhf = _capture_build_property_uhf("uhf")
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    ep_uhf = compute_molecular_integrals_from_fcidump.fn(str(FCIDUMP_FILE), unrestricted=True)
e_uhf = _captured["uhf_e_tot"]
s2_uhf = _captured["uhf_spin_sq"]
print(f"UHF (energy-ordered) energy: {e_uhf:.6f} Ha   <S^2>={s2_uhf}")
print(f"Delta vs RHF: {(e_uhf - e_rhf) * 1000:.3f} mHa")


13:09:46 [INFO] 

******** <class 'pyscf.scf.uhf.UHF'> ********
method = UHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /var/folders/0n/yh83qytx5db3kywhhb_
... [316753 characters of PySCF internal SCF/stability-analysis/Newton-solver log output suppressed for readability -- this is expected verbose=4 chatter, not an error] ...
107
cycle = 197  norm(lambda1,lambda2) = 0.000780233
cycle = 198  norm(lambda1,lambda2) = 0.00078014
cycle = 199  norm(lambda1,lambda2) = 0.000781223
cycle = 200  norm(lambda1,lambda2) = 0.000778637



UHF (energy-ordered) energy: -116.367071 Ha   <S^2>=4.3853025968587165
Delta vs RHF: -161.255 mHa


## Step 3 -- BS-UHF (broken-symmetry, localized guess)

To reach a genuinely unrestricted, lower-energy solution we seed UHF with an atom-localized
(Noodleman) spin guess instead of the energy-ordered default. The guess is specified as an
`AF_GROUPS` JSON dict of orbital-index fragments, set via the `FE4S4_AF_GROUPS` env var.

**Naming quirk, up front** (see the tutorial's "A naming quirk" section for the full explanation):
this knob, and its siblings `FE4S4_AF_POL` / `FE4S4_AF_FREE_S`, are named `FE4S4_*` because they
were first added for the Fe4S4 study. The mechanism is fully generic -- the code only consumes the
JSON/value you give it, with zero Fe-specific logic; only the env var *name* is Fe4S4-branded. You
use the exact same variable names for any molecule.

The fragment ranges below are this repo's own verified Fe2S2 AF-groups spec, taken directly from
`algorithms/sbd/sweep/run_fe2s2_uhf_sample_roquo.sh` (the shipped Fe2S2 UHF sampling launcher):

| fragment | orbital indices | role |
| --- | --- | --- |
| `l1` | 0, 1 | bridging-ligand orbitals (closed) |
| `fe1` | 2, 3, 4, 5, 6 | Fe #1 3d block -- spin-up (`up`) |
| `s` | 7, 8, 9, 10, 11, 12 | bridging-S 3p block (closed) |
| `fe2` | 13, 14, 15, 16, 17 | Fe #2 3d block -- spin-down (`down`) |
| `l2` | 18, 19 | bridging-ligand orbitals (closed) |

Every fragment not listed in `up`/`down`/`free` is treated as closed (doubly occupied); `up`/`down`
fragments get the `0.5 +/- 0.5*pol` alpha/beta split described in the tutorial (`pol=1.0` default
here, i.e. a full localized split).


In [5]:
af_groups = {
    "l1": [0, 1],
    "fe1": [2, 3, 4, 5, 6],
    "s": [7, 8, 9, 10, 11, 12],
    "fe2": [13, 14, 15, 16, 17],
    "l2": [18, 19],
    "up": ["fe1"],
    "down": ["fe2"],
}
os.environ["FE4S4_AF_GROUPS"] = json.dumps(af_groups)

_chem._build_property_uhf = _capture_build_property_uhf("bsuhf")
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    ep_bsuhf = compute_molecular_integrals_from_fcidump.fn(str(FCIDUMP_FILE), unrestricted=True)
e_bsuhf = _captured["bsuhf_e_tot"]
s2_bsuhf = _captured["bsuhf_spin_sq"]
os.environ.pop("FE4S4_AF_GROUPS", None)
_chem._build_property_uhf = _orig_build_property_uhf

print(f"BS-UHF energy: {e_bsuhf:.6f} Ha   <S^2>={s2_bsuhf}")
print(f"Delta vs energy-ordered UHF: {(e_bsuhf - e_uhf) * 1000:.3f} mHa")
if e_bsuhf < e_uhf:
    print("BS-UHF is lower, as expected for this well-separated two-Fe-center system.")
else:
    print("BS-UHF did not land below energy-ordered UHF for this run -- reporting the numbers as measured.")


13:09:48 [INFO] 

******** <class 'pyscf.scf.uhf.UHF'> ********
method = UHF
initial guess = minao
damping factor = 0
level_shift factor = 0
DIIS = <class 'pyscf.scf.diis.CDIIS'>
diis_start_cycle = 1
diis_space = 8
diis_damp = 0
SCF conv_tol = 1e-09
SCF conv_tol_grad = None
SCF max_cycles = 50
direct_scf = True
direct_scf_tol = 1e-13
chkfile to save SCF result = /var/folders/0n/yh83qytx5db3kywhhb_
... [300298 characters of PySCF internal SCF/stability-analysis/Newton-solver log output suppressed for readability -- this is expected verbose=4 chatter, not an error] ...
12517e-06
cycle = 26  norm(lambda1,lambda2) = 2.63556e-06
cycle = 27  norm(lambda1,lambda2) = 1.74177e-06
cycle = 28  norm(lambda1,lambda2) = 1.23239e-06
cycle = 29  norm(lambda1,lambda2) = 8.81e-07



BS-UHF energy: -116.512692 Ha   <S^2>=4.893151067394783
Delta vs energy-ordered UHF: -145.621 mHa
BS-UHF is lower, as expected for this well-separated two-Fe-center system.


## Comparison

All three numbers above, side by side (nothing here is recomputed -- this just formats the values
already printed).


In [6]:
rows = [
    ("RHF", e_rhf, None),
    ("UHF (energy-ordered)", e_uhf, s2_uhf),
    ("BS-UHF (localized)", e_bsuhf, s2_bsuhf),
]
header = f"{'method':<24} {'energy (Ha)':>16} {'<S^2>':>10}"
print(header)
print("-" * len(header))
for name, e, s2 in rows:
    s2_str = f"{s2:.4f}" if s2 is not None else "--"
    print(f"{name:<24} {e:>16.6f} {s2_str:>10}")


method                        energy (Ha)      <S^2>
----------------------------------------------------
RHF                           -116.205816         --
UHF (energy-ordered)          -116.367071     4.3853
BS-UHF (localized)            -116.512692     4.8932


## Step 4 -- LUCJ circuit build

Using the BS-UHF reference, we build LUCJ ansatz parameters from its UCCSD `t2` amplitudes via
`sbd.lucj.initialize_ucj_parameters` (the exact function `run_local_uhf.py` uses for its UHF path)
and turn them into a circuit via `sbd.lucj.create_lucj_circuit`. Both are Prefect `@task`s, so we
again call `.fn(...)`.

To get a realistic ISA (basis-gate, hardware-connectivity) circuit with no real IBM Quantum access,
we transpile against a `GenericBackendV2` fake backend -- the same fake-backend pattern this
repo's own layout tests (`tests/test_error_aware_layout.py`) use for hardware-free transpilation
checks.


In [7]:
norb = ep_bsuhf.num_orbitals
num_elec_a, num_elec_b = ep_bsuhf.num_electrons
aa_indices = [(p, p + 1) for p in range(norb - 1)]
ab_indices = [(p, p) for p in range(0, norb, 4)]

ucj_params = initialize_ucj_parameters.fn(
    elec_props=ep_bsuhf,
    aa_indices=aa_indices,
    ab_indices=ab_indices,
    num_walkers=1,
    randomization_factor=0.2,
    n_lucj_layers=1,
)
vir_circuit = create_lucj_circuit.fn(
    ucj_parameter=ucj_params[0],
    elec_props=ep_bsuhf,
    aa_indices=aa_indices,
    ab_indices=ab_indices,
    n_lucj_layers=1,
    use_reset_mitigation=False,
)
print(f"Virtual LUCJ circuit: {vir_circuit.num_qubits} qubits, {len(vir_circuit.data)} instructions")


/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/scipy/linalg/_matfuncs_inv_ssq.py:881: RuntimeWarning: divide by zero encountered in dot
  return Z.dot(U).dot(ZH)
/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/scipy/linalg/_matfuncs_inv_ssq.py:881: RuntimeWarning: overflow encountered in dot
  return Z.dot(U).dot(ZH)
/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/scipy/linalg/_matfuncs_inv_ssq.py:881: RuntimeWarning: invalid value encountered in dot
  return Z.dot(U).dot(ZH)
13:09:48 [INFO] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/kunalkumar/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file

Virtual LUCJ circuit: 40 qubits, 42 instructions


/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: divide by zero encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)
/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: overflow encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)
/Users/kunalkumar/Projects/riken/roquo/qcsc-prefect/algorithms/sbd/.venv/lib/python3.12/site-packages/ffsim/linalg/predicates.py:83: RuntimeWarning: invalid value encountered in matmul
  return m == n and np.allclose(mat @ mat.T.conj(), np.eye(m), rtol=rtol, atol=atol)


In [8]:
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

backend = GenericBackendV2(num_qubits=vir_circuit.num_qubits, seed=1)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=42)
isa_circuit = pm.run(vir_circuit)
two_qubit_gate_count = sum(1 for instr in isa_circuit.data if instr.operation.num_qubits == 2)

print(f"ISA circuit: {isa_circuit.num_qubits} qubits")
print(f"Total gates: {len(isa_circuit.data)}")
print(f"Two-qubit gates: {two_qubit_gate_count}")


13:09:55 [INFO] Pass: UnrollCustomDefinitions - 0.08011 (ms)


13:09:55 [INFO] Pass: BasisTranslator - 0.15688 (ms)


13:09:55 [INFO] Pass: ContainsInstruction - 0.02408 (ms)


13:09:55 [INFO] Pass: UnitarySynthesis - 0.00525 (ms)


13:09:55 [INFO] Pass: HighLevelSynthesis - 5.41496 (ms)


...

13:09:55 [INFO] Pass: Depth - 0.38815 (ms)


13:09:55 [INFO] Pass: FixedPoint - 0.00405 (ms)


13:09:55 [INFO] Pass: FixedPoint - 0.00215 (ms)


13:09:55 [INFO] Pass: ContainsInstruction - 0.00691 (ms)


ISA circuit: 40 qubits
Total gates: 17922
Two-qubit gates: 2322


## Step 5 -- Local recovery pass

One small local SQD recovery step, with no real device and no simulator: `generate_bit_array_uniform`
produces deterministic pseudo-random bitstrings -- the exact mechanism behind `quantum_source=
"random"` in the shipped scripts. The rest of the pipeline is identical to the real-hardware path:
`recover_configurations` biases the raw samples toward the mean-field occupancies, `postselect_bitstrings`
keeps only the correct-Hamming-weight (particle-number-conserving) strings, and `subsample_open_shell`
dedups and caps each spin channel independently at `sqd_dim`, producing the alpha/beta CI-string
arrays the SBD solver would diagonalize next.


In [9]:
sqd_dim = 3000
bit_array = generate_bit_array_uniform(num_samples=sqd_dim, num_bits=norb * 2, rand_seed=24)
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

bitstrings, probs = recover_configurations(
    bitstring_matrix=raw_bitstrings,
    probabilities=raw_probs,
    avg_occupancies=ep_bsuhf.initial_occupancy,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    rand_seed=np.random.default_rng(24),
)
bitstrings_post, probs_post = postselect_bitstrings.fn(
    bitstring_matrix=bitstrings,
    probabilities=probs,
    hamming_right=num_elec_a,
    hamming_left=num_elec_b,
)

carryover_a = np.zeros((0, norb), dtype=bool)
carryover_b = np.zeros((0, norb), dtype=bool)
ci_a, ci_b = subsample_open_shell.fn(
    bitstring_matrix=bitstrings_post,
    probabilities=probs_post,
    carryover_a=carryover_a,
    carryover_b=carryover_b,
    subspace_dim=sqd_dim,
    norb=norb,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    rng=np.random.default_rng(24),
)
print(f"sqd_dim={sqd_dim}  ->  {len(ci_a)} alpha strings, {len(ci_b)} beta strings"
      f"  (net CI matrix {len(ci_a)} x {len(ci_b)} = {len(ci_a) * len(ci_b)})")


sqd_dim=3000  ->  54 alpha strings, 54 beta strings  (net CI matrix 54 x 54 = 2916)


## Step 6 -- Excitation-rank breakdown

Per-spin unique-determinant counts by excitation order relative to the Hartree-Fock reference
string, in the same `S=.. D=.. T=.. Q=.. ≥5=..` format the tutorial's Step 7 (and every shipped
launcher's `[diag] recovery ...` log line) uses. This compares each **unique** alpha/beta bitstring's
popcount-XOR against the HF string -- a per-spin unique-string count, not a naive per-shot joint
alpha+beta excitation count (that was a documented bug in an earlier session: it inflates high-
excitation counts and hides how saturated the low orders already are).

`sbd.sqd._compute_excitation_counts` is the corrected, shipped implementation; this cell just calls
it on `ci_a`/`ci_b` directly.


In [10]:
counts_a = _compute_excitation_counts(ci_a, num_elec_a)
counts_b = _compute_excitation_counts(ci_b, num_elec_b)

def _fmt(counts):
    return (f"n={counts['n_total']} unique={counts['n_unique']} HF={counts['HF']} "
            f"S={counts['S']} D={counts['D']} T={counts['T']} Q={counts['Q']} "
            f"≥5={counts['high5']} max_exc={counts['max_exc']}")

print("alpha:", _fmt(counts_a))
print("beta :", _fmt(counts_b))

t_notebook_total = time.perf_counter() - t_notebook_start
print(f"\nNotebook wall-clock (from Setup cell): {t_notebook_total:.1f}s")


alpha: n=54 unique=54 HF=1 S=5 D=18 T=20 Q=10 ≥5=0 max_exc=4
beta : n=54 unique=54 HF=1 S=9 D=15 T=23 Q=5 ≥5=1 max_exc=5

Notebook wall-clock (from Setup cell): 13.9s


## How this scales

Everything above used a tiny `sqd_dim` and a single recovery step so the whole notebook runs in
seconds. A real production run differs in three ways:

1. **Sample from real hardware** instead of `generate_bit_array_uniform` (`quantum_source=
   "real-device"` in the shipped flow).
2. **Persist the pool** instead of resampling every time.
3. **Hand the persisted pool to [`run_recover.py`](run_recover.py)** with many more recovery steps
   and a much larger `sqd_dim`, sized per
   [`docs/reference/hpc_resource_sizing.md`](../../docs/reference/hpc_resource_sizing.md), on
   Fugaku or ROQUO.

See [`docs/tutorials/run_uhf_bsuhf_any_molecule.md`](../../docs/tutorials/run_uhf_bsuhf_any_molecule.md)
for the full path from here to a checkpointed, multi-node deep run.
